In [1]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [2]:
from pyspark.sql import SparkSession
# For Linear Interpolation
from pyspark.sql import Window
import pyspark.sql.functions as F

from pyspark.sql.functions import dayofweek, count
from pyspark.sql.functions import explode, sequence, to_date, min, max, lit

##### Data Ingestion


Initialize a Spark session

In [ ]:
spark = SparkSession.builder.appName("SMS_Spam_Detection").getOrCreate()

Read the dataset into a PySpark 

In [4]:
df = spark.read.csv('all_stocks_5yr.csv', header=True, inferSchema=True)
df.show(5)

+----------+-----+-----+-----+-----+--------+----+
|      date| open| high|  low|close|  volume|Name|
+----------+-----+-----+-----+-----+--------+----+
|2013-02-08|15.07|15.12|14.63|14.75| 8407500| AAL|
|2013-02-11|14.89|15.01|14.26|14.46| 8882000| AAL|
|2013-02-12|14.45|14.51| 14.1|14.27| 8126000| AAL|
|2013-02-13| 14.3|14.94|14.25|14.66|10259500| AAL|
|2013-02-14|14.94|14.96|13.16|13.99|31879900| AAL|
+----------+-----+-----+-----+-----+--------+----+
only showing top 5 rows


Display schema and sample rows

In [5]:
print("schema:")
df.printSchema()
print("sample rows:")
df.show(5)

schema:
root
 |-- date: date (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: integer (nullable = true)
 |-- Name: string (nullable = true)

sample rows:
+----------+-----+-----+-----+-----+--------+----+
|      date| open| high|  low|close|  volume|Name|
+----------+-----+-----+-----+-----+--------+----+
|2013-02-08|15.07|15.12|14.63|14.75| 8407500| AAL|
|2013-02-11|14.89|15.01|14.26|14.46| 8882000| AAL|
|2013-02-12|14.45|14.51| 14.1|14.27| 8126000| AAL|
|2013-02-13| 14.3|14.94|14.25|14.66|10259500| AAL|
|2013-02-14|14.94|14.96|13.16|13.99|31879900| AAL|
+----------+-----+-----+-----+-----+--------+----+
only showing top 5 rows


##### Data Cleaning Preprocessing

Handling Missing Values

In [6]:
df.createOrReplaceTempView("stocks") # Create a temporary view for SQL queries

In [7]:
null_analysis_sql = spark.sql("""
    SELECT 
        SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS null_date,
        SUM(CASE WHEN open IS NULL THEN 1 ELSE 0 END) AS null_open,
        SUM(CASE WHEN high IS NULL THEN 1 ELSE 0 END) AS null_high,
        SUM(CASE WHEN low IS NULL THEN 1 ELSE 0 END) AS null_low,
        SUM(CASE WHEN close IS NULL THEN 1 ELSE 0 END) AS null_close,
        SUM(CASE WHEN volume IS NULL THEN 1 ELSE 0 END) AS null_volume,
        SUM(CASE WHEN Name IS NULL THEN 1 ELSE 0 END) AS null_Name,
        COUNT(*) AS total_rows
    FROM stocks
""")
null_analysis_sql.show()

+---------+---------+---------+--------+----------+-----------+---------+----------+
|null_date|null_open|null_high|null_low|null_close|null_volume|null_Name|total_rows|
+---------+---------+---------+--------+----------+-----------+---------+----------+
|        0|       11|        8|       8|         0|          0|        0|    619040|
+---------+---------+---------+--------+----------+-----------+---------+----------+



Fill Missing Values (Forward + Backward)

In [8]:
window_spec = Window.partitionBy("Name").orderBy("date")

def fill_nulls(df, column_name):
    # Forward Fill (from the previous value)
    df = df.withColumn(
        column_name, 
        F.last(column_name, True).over(window_spec.rowsBetween(Window.unboundedPreceding, 0))
    )
    # Backward Fill (for cases at the beginning of the data – takes from the next value)
    df = df.withColumn(
        column_name, 
        F.first(column_name, True).over(window_spec.rowsBetween(0, Window.unboundedFollowing))
    )
    return df


for col_name in ["open", "high", "low"]:
    df = fill_nulls(df, col_name)


df.createOrReplaceTempView("stocks")

spark.sql("SELECT COUNT(*) FROM stocks WHERE open IS NULL").show()

+--------+
|count(1)|
+--------+
|       0|
+--------+



Handling Duplicates

In [9]:
duplicate_check = spark.sql("""
    SELECT date, Name, COUNT(*) as occurrence
    FROM stocks
    GROUP BY date, Name
    HAVING COUNT(*) > 1
""")

duplicate_check.show()
print(f"Number of duplicate: {duplicate_check.count()}")

+----+----+----------+
|date|Name|occurrence|
+----+----+----------+
+----+----+----------+



Number of duplicate: 0


Handling Outliers

In [10]:
quantiles = df.approxQuantile("volume", [0.25, 0.75], 0.01)

# IQR = Q3 -Q1
IQR = quantiles[1] - quantiles[0]

# Upper bound = Q3 + 1.5 * IQR
upper_bound = quantiles[1] + 1.5 * IQR
# Lower bound = Q1 - 1.5 * IQR
lower_bound = quantiles[0] - 1.5 * IQR

outliers_df = df.filter((F.col("volume") > upper_bound) | (F.col("volume") < lower_bound))
outliers_count = outliers_df.count()
total_count = df.count()

print(f"Total Rows: {total_count}")
print(f"Lower Bound: {lower_bound}")
print(f"Upper Bound: {upper_bound}")
print(f"Number of Outliers detected: {outliers_count}")
print(f"Percentage of Outliers: {(outliers_count / total_count) * 100:.2f}%")

Total Rows: 619040
Lower Bound: -3668898.0
Upper Bound: 8934982.0
Number of Outliers detected: 61103
Percentage of Outliers: 9.87%


##### Resampling & Frequency

Proof of Weekend Closure (Stock Market is Closed on Saturdays & Sundays)

In [11]:
df_check = df.withColumn("day_num", dayofweek("date")) # 1 = Sunday, 2 = Monday, ..., 7 = Saturday

weekend_analysis = df_check.groupBy("day_num").agg(count("*").alias("record_count")).orderBy("day_num")

weekend_analysis.show()

+-------+------------+
|day_num|record_count|
+-------+------------+
|      2|      115998|
|      3|      127831|
|      4|      127377|
|      5|      123923|
|      6|      123911|
+-------+------------+



In [12]:
# 1. Get the global start and end dates
date_range = df.select(min("date").alias("min_date"), max("date").alias("max_date")).collect()[0]

# 2. Create a DataFrame with all possible dates between min and max
# Use 'sequence' to generate every single day
all_dates_df = spark.sql(f"""
    SELECT explode(sequence(to_date('{date_range['min_date']}'), to_date('{date_range['max_date']}'), interval 1 day)) as date
""")

# 3. Get all unique company names
names_df = df.select("Name").distinct()

# 4. Cross Join to create a (Date x Name) template (Full Grid)
full_grid_df = names_df.crossJoin(all_dates_df)

In [13]:
# 1. Left join our data with the full grid
df_resampled = full_grid_df.join(df, ["Name", "date"], "left")

# 2. Sorting to prepare for filling
df_resampled = df_resampled.orderBy("Name", "date")

# 3. Decision for Gap Filling:
# For 'close' and 'volume', we use Forward Fill (Last Observation Carried Forward - LOCF)
# Because if the market is closed, the price stays the same as the last trading day, 
# and the volume (demand) is technically 0 for that day.

window_fill = Window.partitionBy("Name").orderBy("date").rowsBetween(Window.unboundedPreceding, 0)

# Fill Prices with Forward Fill
for c in ["open", "high", "low", "close"]:
    df_resampled = df_resampled.withColumn(c, F.last(c, True).over(window_fill))

# Fill Volume with 0 (Since no trading happened on weekends)
df = df_resampled.na.fill({"volume": 0})
 
df_check = df.withColumn("day_num", dayofweek("date")) # 1 = Sunday, 2 = Monday, ..., 7 = Saturday
weekend_analysis = df_check.groupBy("day_num").agg(count("*").alias("record_count")).orderBy("day_num")
weekend_analysis.show()


+-------+------------+
|day_num|record_count|
+-------+------------+
|      1|      131805|
|      2|      131805|
|      3|      131805|
|      4|      131805|
|      5|      131300|
|      6|      131805|
|      7|      131805|
+-------+------------+

